## Event Validation

written by Isobel Mawby (i.mawby1@lancaster.ac.uk)

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Imports
</div>

In [ ]:
import random
import uproot
import numpy as np
import math
import matplotlib.pyplot as plt
import awkward as ak

%matplotlib widget
from termcolor import colored, cprint

import Definitions
import ValidationFunc
import EventValidationFunc

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Config
</div>

In [ ]:
SHOW_PLOTS = True

<div class="alert alert-block alert-info" style="font-size: 18px;">
    File
</div>

In [ ]:
file_name = "/Users/isobel/Desktop/DUNE/2026/PandoraValidation/files/ValidationVis.root"
plot_dir = '/Users/isobel/Desktop/DUNE/2026/PandoraValidation/EventValPlots/'

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Lets open the file...
</div>

In [ ]:
file = uproot.open(file_name)

In [ ]:
#event_tree.keys()

In [ ]:
event_tree = file['EventTree']

event_branches = event_tree.arrays(['Run', 'Subrun', 'Event', 'MCInt_IsCC', 'MCNu_PDG', 'MCNu_Energy', 'MCNu_VisEnergy',
                                    'MCNu_VertexX', 'MCNu_VertexY', 'MCNu_VertexZ',
                                    'RecoNu_VertexX', 'RecoNu_VertexY', 'RecoNu_VertexZ',
                                    'RecoNu_VertexAcc_Pass2'], library="ak")

In [ ]:
flat_trueEnergy = ak.to_numpy(event_branches['MCNu_Energy'])
flat_visEnergy = ak.to_numpy(event_branches['MCNu_VisEnergy'])

true_nu_energy_var = ValidationFunc.PlotVar('MCNu_Energy', 'True Nu Energy', 'Frac. of MCNu', [0,10.0], 20)
true_vis_nu_energy_var = ValidationFunc.PlotVar('MCNu_VisEnergy', 'True Vis Nu Energy', 'Frac. of MCNu', [0,10.0], 20)

fig, ax = plt.subplots()
ax.hist2d(flat_trueEnergy, flat_visEnergy, bins=[true_nu_energy_var.n_bins, true_vis_nu_energy_var.n_bins], range=[true_nu_energy_var.range, true_vis_nu_energy_var.range])
ax.set_xlabel(true_nu_energy_var.x_label)
ax.set_ylabel(true_vis_nu_energy_var.x_label)
plt.show()




<div class="alert alert-block alert-info" style="font-size: 18px;">
    Get detector edges
</div>

In [ ]:
detector = {'X': (-363, 363), 'Y': (-604, 604), 'Z': (0, 1393)}
true_nu_vtx_boundary = {'X': (ak.min(event_branches['MCNu_VertexX']), ak.max(event_branches['MCNu_VertexX'])),
                        'Y': (ak.min(event_branches['MCNu_VertexY']), ak.max(event_branches['MCNu_VertexY'])),
                        'Z': (ak.min(event_branches['MCNu_VertexZ']), ak.max(event_branches['MCNu_VertexZ']))}

In [ ]:



# # Vertex distributions
# fig_true, ax_true = plt.subplots(ncols=3, figsize=(17, 6))
# EventValidationFunc.PlotVertices(event_branches, 'MCNu_Vertex', 'violet', detector, true_nu_vtx_boundary, (ak.ones_like(event_branches['MCNu_VertexX']) == 1), ax_true)
# fig_true.savefig(f'{plot_dir}MCNuVertex.pdf', bbox_inches='tight')
# fig_reco, ax_reco = plt.subplots(ncols=3, figsize=(17, 6))
# EventValidationFunc.PlotVertices(event_branches, 'RecoNu_Vertex', 'blue', detector, true_nu_vtx_boundary, (ak.ones_like(event_branches['MCNu_VertexX']) == 1), ax_reco)
# fig_reco.savefig(f'{plot_dir}RecoNuVertex.pdf', bbox_inches='tight')

# # Out of detector i.e. reconstruction failures
# out_of_detector = (event_branches['RecoNu_VertexZ'] < true_nu_vtx_boundary['Z'][0]) | (event_branches['RecoNu_VertexZ'] > true_nu_vtx_boundary['Z'][1]) | (event_branches['RecoNu_VertexX'] < true_nu_vtx_boundary['X'][0]) | (event_branches['RecoNu_VertexX'] > true_nu_vtx_boundary['X'][1]) | (event_branches['RecoNu_VertexY'] < true_nu_vtx_boundary['Y'][0]) | (event_branches['RecoNu_VertexY'] > true_nu_vtx_boundary['Y'][1])
# fig_out_of_detector_true, ax_out_of_detector_true = plt.subplots(ncols=3, figsize=(17, 6))
# EventValidationFunc.PlotVertices(event_branches, 'MCNu_Vertex', 'violet', detector, true_nu_vtx_boundary, out_of_detector, ax_out_of_detector_true)
# fig_out_of_detector_true.savefig(f'{plot_dir}MCNuVertex_OutOfDetector.pdf', bbox_inches='tight')
# fig_out_of_detector_reco, ax_out_of_detector_reco = plt.subplots(ncols=3, figsize=(17, 6))
# EventValidationFunc.PlotVertices(event_branches, 'RecoNu_Vertex', 'blue', detector, true_nu_vtx_boundary, out_of_detector, ax_out_of_detector_reco)
# fig_out_of_detector_reco.savefig(f'{plot_dir}RecoNuVertex_OutOfDetector.pdf', bbox_inches='tight')

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Summary
</div>

In [ ]:
int_masks = Definitions.GetIntMasks(event_branches, pfp_branches={}, broadcast=False)

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Get plots + tables
</div>

In [ ]:
print(ak.__version__)

In [ ]:
nu_true_energy = ValidationFunc.PlotVar('MCNu_Energy', 'True Nu Energy [GeV]', 'Frac. of true neutrino', [0,10.0], 40)
nu_vertex_accuracy = ValidationFunc.PlotVar('RecoNu_VertexAcc_Pass2', 'NuVertexAccuracy [cm]', 'Frac. of reco neutrino', [0,10.0], 50)

nu_vtx_delta_x = ValidationFunc.PlotDiffVar('MCNu_VertexX', 'RecoNu_VertexX', 'TrueX-RecoX [cm]', 'Frac. of reco neutrino', [-10,10.0], 100)
nu_vtx_delta_y = ValidationFunc.PlotDiffVar('MCNu_VertexY', 'RecoNu_VertexY', 'TrueY-RecoY [cm]', 'Frac. of reco neutrino', [-10,10.0], 100)
nu_vtx_delta_z = ValidationFunc.PlotDiffVar('MCNu_VertexZ', 'RecoNu_VertexZ', 'TrueZ-RecoZ [cm]', 'Frac. of reco neutrino', [-10,10.0], 100)

# MCP branch variables to plot
MCP_plotting_vars = [nu_true_energy]

# BM branch variables to plot
BM_plotting_vars = [nu_vertex_accuracy]

# Diff variables
diff_plotting_vars = [nu_vtx_delta_x, nu_vtx_delta_y, nu_vtx_delta_z]

# MCP branch variable plots
MCP_var_plots = [plt.subplots(ncols=len(Definitions.ints), nrows=1, figsize=(16, 5)) for _ in MCP_plotting_vars]

# BM branch variable plots
BM_var_plots = [plt.subplots(ncols=len(Definitions.ints), nrows=1, figsize=(16, 5)) for _ in BM_plotting_vars]

# Diff branch variable plots
diff_var_plots = [plt.subplots(ncols=len(Definitions.ints), nrows=1, figsize=(16, 5)) for _ in diff_plotting_vars]

# Vertex dR plots
vertex_dr_plots_true = plt.subplots(ncols=len(Definitions.ints), nrows=1, figsize=(16, 5))
vertex_dr_plots_reco = plt.subplots(ncols=len(Definitions.ints), nrows=1, figsize=(16, 5))

for int_type in Definitions.ints :
    int_mask = int_masks[int_type]

    target_mask = int_mask
    reco_mask = target_mask & (event_branches['RecoNu_VertexZ'] > -9000)

    # Plot MCP_var distributions
    for i_var in range(len(MCP_plotting_vars)) :
        fig, axes = MCP_var_plots[i_var]
        ax = axes[int_type]
        ValidationFunc.ConfigurePlot(fig, ax, int_type, -1, -1, MCP_plotting_vars[i_var])                
        ValidationFunc.PlotVariable(target_mask, event_branches, MCP_plotting_vars[i_var], ax, Definitions.int_strings[int_type], Definitions.int_color[int_type])
        if not SHOW_PLOTS :
            plt.close(fig)

    # Plot BM_var distributions
    for i_var in range(len(BM_plotting_vars)) :
        fig, axes = BM_var_plots[i_var]
        ax = axes[int_type]
        ValidationFunc.ConfigurePlot(fig, ax, int_type, -1, -1, BM_plotting_vars[i_var])
        ValidationFunc.PlotVariable(reco_mask, event_branches, BM_plotting_vars[i_var], ax, Definitions.int_strings[int_type], Definitions.int_color[int_type])
        if not SHOW_PLOTS :
            plt.close(fig)      

    # Plot diff_vars
    for i_var in range(len(diff_plotting_vars)) :
        fig, axes = diff_var_plots[i_var]
        ax = axes[int_type]
        ValidationFunc.ConfigurePlot(fig, ax, int_type, -1, -1, diff_plotting_vars[i_var])
        ValidationFunc.PlotDiffVariable(reco_mask, event_branches, diff_plotting_vars[i_var], ax, Definitions.int_strings[int_type], Definitions.int_color[int_type])
        if not SHOW_PLOTS :
            plt.close(fig)

    # Plot vertex dR plots
    fig, axes = vertex_dr_plots_true
    EventValidationFunc.PlotVertexCumulativeDR(event_branches, target_mask, True,  Definitions.int_strings[int_type], Definitions.int_color[int_type], axes[int_type], fig)
    if not SHOW_PLOTS :
        plt.close(fig)
    fig, axes = vertex_dr_plots_reco
    EventValidationFunc.PlotVertexCumulativeDR(event_branches, reco_mask, False,  Definitions.int_strings[int_type], Definitions.int_color[int_type], axes[int_type], fig)
    if not SHOW_PLOTS :
        plt.close(fig)        

# Save MCP_var distributions
for i_var in range(len(MCP_plotting_vars)) :
    fig, _ = MCP_var_plots[i_var]
    file_name = f'{MCP_plotting_vars[i_var].tree_name}'
    fig.savefig(f'{plot_dir}{file_name}.pdf', bbox_inches='tight')

# Save BM_var distributions
for i_var in range(len(BM_plotting_vars)) :
    fig, _ = BM_var_plots[i_var]
    file_name = f'{BM_plotting_vars[i_var].tree_name}'
    fig.savefig(f'{plot_dir}{file_name}.pdf', bbox_inches='tight')     

# Save diff var distributions
for i_var in range(len(diff_plotting_vars)) :
    fig, _ = diff_var_plots[i_var]
    file_name = f'{diff_plotting_vars[i_var].true_tree_name}-{diff_plotting_vars[i_var].reco_tree_name}'
    fig.savefig(f'{plot_dir}{file_name}.pdf', bbox_inches='tight')    

fig, axes = vertex_dr_plots_true
fig.savefig(f'{plot_dir}CumulativeDR_True.pdf', bbox_inches='tight')  
fig, axes = vertex_dr_plots_reco
fig.savefig(f'{plot_dir}CumulativeDR_Reco.pdf', bbox_inches='tight')  